# 🚀 Анализ и прогнозирование криптовалют

Демонстрация системы анализа криптовалютных рынков с использованием Binance API и машинного обучения.

## 📦 Импорт библиотек и настройка

In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
from pathlib import Path

# Добавляем путь к модулям проекта
sys.path.append('..')

# Импорт наших модулей
from src.utils.config import config
from src.data.binance_client import BinanceClient
from src.data.data_collector import DataCollector
from src.analysis.technical_indicators import TechnicalIndicators
from src.prediction.ml_models import CryptoPricePredictor

# Настройки
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("✅ Модули загружены успешно")

## 🔧 Проверка конфигурации системы

In [ ]:
# Показать конфигурацию
config.print_config_summary()

# Проверка валидации
validation_results = config.validate_config()
print("\n🔍 Результаты валидации:")
for check, result in validation_results.items():
    status = "✅" if result else "❌"
    print(f"   {status} {check}: {result}")

## 🌐 Подключение к Binance API

In [ ]:
# Инициализация клиента
client = BinanceClient()

# Проверка подключения
if client.test_connectivity():
    print("✅ Подключение к Binance API установлено")
    
    # Получение топ символов
    top_symbols = client.get_top_symbols(limit=10)
    print(f"\n📊 Топ-10 торговых пар по объему:")
    for i, symbol in enumerate(top_symbols, 1):
        print(f"{i:2d}. {symbol}")
else:
    print("❌ Ошибка подключения к Binance API")

## 📈 24-часовая статистика рынка

In [ ]:
# Получение 24ч статистики для основных криптовалют
major_symbols = config.get_symbols_by_category('major')
ticker_data = []

for symbol in major_symbols:
    try:
        ticker = client.get_24hr_ticker(symbol)
        if not ticker.empty:
            data = ticker.iloc[0]
            ticker_data.append({
                'Symbol': symbol,
                'Price': float(data['lastPrice']),
                'Change %': float(data['priceChangePercent']),
                'Volume': float(data['volume']),
                'High': float(data['highPrice']),
                'Low': float(data['lowPrice'])
            })
    except Exception as e:
        print(f"⚠️ Не удалось получить данные для {symbol}: {e}")

if ticker_data:
    ticker_df = pd.DataFrame(ticker_data)
    
    print("📊 24-часовая статистика основных криптовалют:")
    print(ticker_df.to_string(index=False, float_format='%.2f'))
    
    # Визуализация изменения цен
    fig = px.bar(ticker_df, x='Symbol', y='Change %', 
                 title='24-часовое изменение цен (%)',
                 color='Change %',
                 color_continuous_scale=['red', 'green'])
    fig.show()
else:
    print("❌ Не удалось получить данные о ценах")

## 📊 Загрузка исторических данных

In [ ]:
# Выберем символ для анализа
SYMBOL = 'BTCUSDT'
INTERVAL = '1h'
DAYS_BACK = 90

print(f"📥 Загрузка данных для {SYMBOL} с интервалом {INTERVAL}...")

# Загрузка данных
from datetime import datetime, timedelta

end_time = datetime.now()
start_time = end_time - timedelta(days=DAYS_BACK)

df = client.get_historical_klines(
    symbol=SYMBOL,
    interval=INTERVAL,
    start_time=start_time.isoformat(),
    end_time=end_time.isoformat(),
    limit=1000
)

if not df.empty:
    print(f"✅ Загружено {len(df)} записей")
    print(f"📅 Период: {df.index[0]} - {df.index[-1]}")
    print(f"💰 Текущая цена: ${df['close'].iloc[-1]:.2f}")
    
    # Показать первые несколько записей
    print("\n📋 Первые 5 записей:")
    display(df[['open', 'high', 'low', 'close', 'volume']].head())
else:
    print("❌ Не удалось загрузить данные")

## 📊 Визуализация ценового графика

In [ ]:
if not df.empty:
    # Создание интерактивного графика свечей
    fig = go.Figure()
    
    # Добавление свечей
    fig.add_trace(go.Candlestick(
        x=df.index,
        open=df['open'],
        high=df['high'],
        low=df['low'],
        close=df['close'],
        name=SYMBOL
    ))
    
    fig.update_layout(
        title=f'{SYMBOL} - Candlestick Chart ({INTERVAL})',
        yaxis_title='Цена (USDT)',
        xaxis_title='Время',
        height=600
    )
    
    fig.show()
    
    # График объема торгов
    fig2 = go.Figure()
    fig2.add_trace(go.Scatter(
        x=df.index,
        y=df['volume'],
        mode='lines',
        fill='tonexty',
        name='Объем'
    ))
    
    fig2.update_layout(
        title=f'{SYMBOL} - Объем торгов',
        yaxis_title='Объем',
        xaxis_title='Время',
        height=400
    )
    
    fig2.show()

## 🔍 Расчет технических индикаторов

In [ ]:
if not df.empty:
    print("🔍 Расчет технических индикаторов...")
    
    # Расчет всех индикаторов
    df_indicators = TechnicalIndicators.calculate_all_indicators(df)
    
    print(f"✅ Рассчитано {len(df_indicators.columns)} индикаторов")
    
    # Показать последние значения основных индикаторов
    latest = df_indicators.iloc[-1]
    
    print("\n📊 Последние значения технических индикаторов:")
    print(f"💰 Цена закрытия: ${latest['close']:.2f}")
    
    if 'RSI' in latest:
        rsi = latest['RSI']
        rsi_signal = "Перекупленность" if rsi > 70 else "Перепроданность" if rsi < 30 else "Нейтрально"
        print(f"📈 RSI: {rsi:.2f} - {rsi_signal}")
    
    if 'SMA_20' in latest and 'SMA_50' in latest:
        sma_signal = "Бычий" if latest['SMA_20'] > latest['SMA_50'] else "Медвежий"
        print(f"📊 SMA Signal: {sma_signal} (SMA20: {latest['SMA_20']:.2f}, SMA50: {latest['SMA_50']:.2f})")
    
    if 'MACD' in latest and 'MACD_Signal' in latest:
        macd_signal = "Бычий" if latest['MACD'] > latest['MACD_Signal'] else "Медвежий"
        print(f"⚡ MACD Signal: {macd_signal}")
    
    if 'BB_Upper' in latest and 'BB_Lower' in latest:
        bb_width = ((latest['BB_Upper'] - latest['BB_Lower']) / latest['close']) * 100
        print(f"🎯 Bollinger Bands Width: {bb_width:.2f}%")
    
    # Показать корреляцию индикаторов
    indicator_columns = ['close', 'RSI', 'MACD', 'SMA_20', 'SMA_50', 'BB_Upper', 'BB_Lower']
    available_columns = [col for col in indicator_columns if col in df_indicators.columns]
    
    if len(available_columns) > 2:
        correlation_matrix = df_indicators[available_columns].corr()
        
        plt.figure(figsize=(10, 8))
        sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0,
                   square=True, fmt='.2f', cbar_kws={'shrink': 0.8})
        plt.title('Корреляция технических индикаторов')
        plt.tight_layout()
        plt.show()

## 📈 Интерактивные графики индикаторов

In [ ]:
if not df.empty and 'df_indicators' in locals():
    # Создание сложного графика с несколькими подграфиками
    fig = make_subplots(
        rows=4, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.05,
        subplot_titles=('Цена и Bollinger Bands', 'RSI', 'MACD', 'Объем'),
        row_heights=[0.4, 0.2, 0.2, 0.2]
    )
    
    # График 1: Цена и Bollinger Bands
    fig.add_trace(go.Scatter(
        x=df_indicators.index, y=df_indicators['close'],
        name='Close', line=dict(color='blue', width=2)
    ), row=1, col=1)
    
    if 'BB_Upper' in df_indicators.columns:
        fig.add_trace(go.Scatter(
            x=df_indicators.index, y=df_indicators['BB_Upper'],
            name='BB Upper', line=dict(color='red', dash='dash')
        ), row=1, col=1)
        
        fig.add_trace(go.Scatter(
            x=df_indicators.index, y=df_indicators['BB_Lower'],
            name='BB Lower', line=dict(color='green', dash='dash'),
            fill='tonexty', fillcolor='rgba(0,100,80,0.1)'
        ), row=1, col=1)
    
    if 'SMA_20' in df_indicators.columns:
        fig.add_trace(go.Scatter(
            x=df_indicators.index, y=df_indicators['SMA_20'],
            name='SMA 20', line=dict(color='orange', width=1)
        ), row=1, col=1)
    
    # График 2: RSI
    if 'RSI' in df_indicators.columns:
        fig.add_trace(go.Scatter(
            x=df_indicators.index, y=df_indicators['RSI'],
            name='RSI', line=dict(color='purple')
        ), row=2, col=1)
        
        # Уровни перекупленности/перепроданности
        fig.add_hline(y=70, line_dash="dash", line_color="red", row=2, col=1)
        fig.add_hline(y=30, line_dash="dash", line_color="green", row=2, col=1)
    
    # График 3: MACD
    if 'MACD' in df_indicators.columns:
        fig.add_trace(go.Scatter(
            x=df_indicators.index, y=df_indicators['MACD'],
            name='MACD', line=dict(color='blue')
        ), row=3, col=1)
        
        if 'MACD_Signal' in df_indicators.columns:
            fig.add_trace(go.Scatter(
                x=df_indicators.index, y=df_indicators['MACD_Signal'],
                name='Signal', line=dict(color='red')
            ), row=3, col=1)
        
        if 'MACD_Histogram' in df_indicators.columns:
            fig.add_trace(go.Bar(
                x=df_indicators.index, y=df_indicators['MACD_Histogram'],
                name='Histogram', opacity=0.6
            ), row=3, col=1)
    
    # График 4: Объем
    fig.add_trace(go.Bar(
        x=df_indicators.index, y=df_indicators['volume'],
        name='Volume', opacity=0.7
    ), row=4, col=1)
    
    fig.update_layout(
        title=f'{SYMBOL} - Технический анализ',
        height=1000,
        showlegend=True
    )
    
    fig.show()

## 🤖 Подготовка данных для машинного обучения

In [ ]:
if 'df_indicators' in locals() and not df_indicators.empty:
    print("🤖 Подготовка данных для машинного обучения...")
    
    # Инициализация предсказателя
    predictor = CryptoPricePredictor()
    
    # Подготовка признаков
    features_df = predictor.prepare_features(df_indicators)
    
    if not features_df.empty:
        print(f"✅ Подготовлено {features_df.shape[1]} признаков из {len(features_df)} записей")
        
        # Показать статистику признаков
        numeric_features = features_df.select_dtypes(include=[np.number])
        
        print("\n📊 Топ-10 признаков с наибольшей корреляцией с ценой закрытия:")
        
        if 'close' in numeric_features.columns:
            correlations = numeric_features.corr()['close'].abs().sort_values(ascending=False)
            top_features = correlations.head(11)[1:11]  # Исключаем саму цену
            
            for i, (feature, corr) in enumerate(top_features.items(), 1):
                print(f"{i:2d}. {feature:<25} : {corr:.3f}")
            
            # Визуализация топ корреляций
            plt.figure(figsize=(12, 6))
            top_features.plot(kind='barh', color='skyblue', edgecolor='navy')
            plt.title('Топ-10 признаков по корреляции с ценой закрытия')
            plt.xlabel('Абсолютная корреляция')
            plt.tight_layout()
            plt.show()
        
        # Создание целевой переменной (будущая цена)
        target = features_df['close'].shift(-1).dropna()  # Цена следующего периода
        features_clean = features_df[:-1]  # Удаляем последнюю строку
        
        print(f"\n🎯 Целевая переменная: {len(target)} значений")
        print(f"🔍 Финальный набор данных: {features_clean.shape[0]} записей, {features_clean.shape[1]} признаков")
        
    else:
        print("❌ Не удалось подготовить признаки")

## 🎯 Демо-обучение простой модели

In [ ]:
if 'features_clean' in locals() and 'target' in locals():
    print("🎯 Обучение демо-модели...")
    
    # Выберем только числовые признаки
    numeric_columns = features_clean.select_dtypes(include=[np.number]).columns
    X = features_clean[numeric_columns].fillna(0)
    y = target
    
    # Обучение только нескольких быстрых моделей для демо
    from sklearn.model_selection import train_test_split
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.linear_model import LinearRegression
    from sklearn.metrics import mean_squared_error, r2_score
    import time
    
    # Разделение данных
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, shuffle=False, random_state=42
    )
    
    models = {
        'Linear Regression': LinearRegression(),
        'Random Forest': RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
    }
    
    results = {}
    
    for name, model in models.items():
        print(f"\n🔄 Обучение {name}...")
        start_time = time.time()
        
        # Обучение
        model.fit(X_train, y_train)
        training_time = time.time() - start_time
        
        # Предсказания
        y_pred = model.predict(X_test)
        
        # Метрики
        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_test, y_pred)
        
        results[name] = {
            'model': model,
            'mse': mse,
            'rmse': rmse,
            'r2': r2,
            'training_time': training_time,
            'predictions': y_pred,
            'actual': y_test
        }
        
        print(f"⏱️  Время обучения: {training_time:.2f}с")
        print(f"📊 RMSE: {rmse:.4f}")
        print(f"📈 R²: {r2:.4f}")
    
    # Сравнение моделей
    comparison_data = []
    for name, result in results.items():
        comparison_data.append({
            'Модель': name,
            'RMSE': result['rmse'],
            'R²': result['r2'],
            'Время (с)': result['training_time']
        })
    
    comparison_df = pd.DataFrame(comparison_data)
    comparison_df = comparison_df.sort_values('RMSE')
    
    print("\n📊 Сравнение моделей:")
    display(comparison_df.round(4))
    
    # Визуализация предсказаний лучшей модели
    best_model_name = comparison_df.iloc[0]['Модель']
    best_result = results[best_model_name]
    
    plt.figure(figsize=(12, 6))
    
    # Показать только последние 100 точек для наглядности
    n_points = min(100, len(best_result['actual']))
    actual_subset = best_result['actual'].iloc[-n_points:]
    pred_subset = best_result['predictions'][-n_points:]
    
    plt.plot(actual_subset.values, label='Фактические цены', alpha=0.7, linewidth=2)
    plt.plot(pred_subset, label=f'Прогнозы ({best_model_name})', alpha=0.7, linewidth=2)
    
    plt.title(f'Сравнение фактических цен и прогнозов ({best_model_name})')
    plt.xlabel('Время (индекс)')
    plt.ylabel('Цена')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Scatter plot для проверки качества
    plt.figure(figsize=(8, 8))
    plt.scatter(best_result['actual'], best_result['predictions'], alpha=0.5)
    
    # Линия идеального предсказания
    min_val = min(best_result['actual'].min(), best_result['predictions'].min())
    max_val = max(best_result['actual'].max(), best_result['predictions'].max())
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', alpha=0.8)
    
    plt.xlabel('Фактические цены')
    plt.ylabel('Прогнозируемые цены')
    plt.title(f'Scatter Plot: Фактические vs Прогнозируемые цены\n{best_model_name} (R² = {best_result["r2"]:.4f})')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
else:
    print("❌ Данные для обучения не подготовлены")

## 🔮 Простое демо-прогнозирование

In [ ]:
if 'results' in locals() and results:
    print("🔮 Создание прогноза на основе последних данных...")
    
    # Получаем последние данные для прогноза
    latest_features = X.tail(1)
    current_price = features_clean['close'].iloc[-1]
    
    print(f"💰 Текущая цена {SYMBOL}: ${current_price:.2f}")
    print("\n🔮 Прогнозы моделей на следующий период:")
    print("-" * 50)
    
    predictions = {}
    
    for model_name, result in results.items():
        model = result['model']
        pred_price = model.predict(latest_features)[0]
        change_pct = ((pred_price - current_price) / current_price) * 100
        
        predictions[model_name] = {
            'price': pred_price,
            'change_pct': change_pct
        }
        
        direction = "📈" if change_pct > 0 else "📉" if change_pct < 0 else "➡️"
        print(f"{direction} {model_name:<20}: ${pred_price:>8.2f} ({change_pct:>+6.2f}%)")
    
    # Средний прогноз
    avg_price = np.mean([p['price'] for p in predictions.values()])
    avg_change = ((avg_price - current_price) / current_price) * 100
    avg_direction = "📈" if avg_change > 0 else "📉" if avg_change < 0 else "➡️"
    
    print("-" * 50)
    print(f"{avg_direction} {'Средний прогноз':<20}: ${avg_price:>8.2f} ({avg_change:>+6.2f}%)")
    
    # Анализ последних технических индикаторов
    if 'df_indicators' in locals():
        latest_indicators = df_indicators.iloc[-1]
        
        print("\n📊 Текущие технические сигналы:")
        print("-" * 40)
        
        if 'RSI' in latest_indicators:
            rsi = latest_indicators['RSI']
            if rsi > 70:
                rsi_signal = "⚠️ Перекупленность"
            elif rsi < 30:
                rsi_signal = "✅ Перепроданность"
            else:
                rsi_signal = "➡️ Нейтрально"
            print(f"RSI ({rsi:.1f}): {rsi_signal}")
        
        if 'MACD' in latest_indicators and 'MACD_Signal' in latest_indicators:
            macd_trend = "📈 Бычий" if latest_indicators['MACD'] > latest_indicators['MACD_Signal'] else "📉 Медвежий"
            print(f"MACD: {macd_trend}")
        
        if 'SMA_20' in latest_indicators and 'SMA_50' in latest_indicators:
            sma_trend = "📈 Восходящий" if latest_indicators['SMA_20'] > latest_indicators['SMA_50'] else "📉 Нисходящий"
            print(f"SMA Тренд: {sma_trend}")
    
    print("\n⚠️ Важно: Это демонстрационные прогнозы, основанные на ограниченных данных.")
    print("    Не используйте их для реальной торговли без дополнительного анализа!")
    
else:
    print("❌ Модели не обучены")

## 📝 Резюме и следующие шаги

In [ ]:
print("🎯 РЕЗЮМЕ АНАЛИЗА")
print("=" * 50)

if 'df' in locals() and not df.empty:
    print(f"📊 Проанализирован символ: {SYMBOL}")
    print(f"📅 Период данных: {df.index[0].strftime('%Y-%m-%d')} - {df.index[-1].strftime('%Y-%m-%d')}")
    print(f"📈 Записей обработано: {len(df)}")
    print(f"💰 Текущая цена: ${df['close'].iloc[-1]:.2f}")
    
    # Статистика изменений
    price_change = ((df['close'].iloc[-1] - df['close'].iloc[0]) / df['close'].iloc[0]) * 100
    volatility = df['close'].pct_change().std() * 100
    
    print(f"📊 Изменение за период: {price_change:+.2f}%")
    print(f"📉 Волатильность: {volatility:.2f}%")

if 'results' in locals() and results:
    print(f"\n🤖 Обучено моделей: {len(results)}")
    best_model = min(results.items(), key=lambda x: x[1]['rmse'])
    print(f"🏆 Лучшая модель: {best_model[0]} (RMSE: {best_model[1]['rmse']:.4f})")

print("\n🚀 СЛЕДУЮЩИЕ ШАГИ:")
print("-" * 30)
print("1. 📊 Собрать больше исторических данных")
print("2. 🔍 Добавить дополнительные технические индикаторы")
print("3. 🧠 Обучить более сложные модели (XGBoost, LSTM)")
print("4. ✅ Провести бэктестинг на исторических данных")
print("5. 📈 Создать систему управления рисками")
print("6. 🔄 Автоматизировать обновление данных")
print("7. 📱 Разработать интерфейс для мониторинга")

print("\n💡 ПОЛЕЗНЫЕ КОМАНДЫ:")
print("-" * 25)
print("# Сбор данных:")
print("python main.py --collect-data --symbols BTCUSDT ETHUSDT")
print("\n# Обучение моделей:")
print("python main.py --train-models --symbol BTCUSDT --lstm")
print("\n# Создание прогноза:")
print("python main.py --predict --symbol BTCUSDT --horizon 24")

print("\n✅ Демонстрация завершена!")